In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import csv
import hashlib
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, fft
from scipy.optimize import linear_sum_assignment
from daspy.basic_tools.visualization import plot as daspy_plot

data_dir = Path(r'D:\projects\das\下午')  # 全部 BIN 文件所在目录
output_root = data_dir / 'demo2_results'  # 按参数与源文件指纹建立独立结果目录
options = dict(
    channel_from=291,                 # 起始通道，1 起算，包含
    channel_to=520,                   # 结束通道，包含
    channel_spacing=4.0,              # 通道间距，m；应使用实际标定值
    window_seconds=60.0,              # 检测窗口长度，s
    step_seconds=30.0,                # 相邻窗口的步长，s
    windows_per_batch=8,              # 每批共享一次预处理，限制重复读取和内存
    frequency_low=0.01,               # 带通下限，Hz
    frequency_high=1.0,               # 带通上限，Hz
    preprocessing_padding=100.0,      # 预处理前后上下文，s；目录两端只能使用已有数据
    fk_padding=5.0,                   # f-k 目标窗口前后上下文，s
    padding_channels=16,              # 通道两端边缘衰减扩展的通道数
    angular_transition=0.15,          # 归一化 f-k 平面中方向掩码的过渡宽度
    minimum_speed_kmh=0.0,            # 表观速度下限，km/h；0 不额外设置慢速截止
    maximum_speed_kmh=120.0,          # 表观速度上限，km/h
    feature_mode='envelope',          # envelope：平滑包络峰；positive：正峰
    smoothing_seconds=0.25,          # 包络高斯平滑 sigma，s
    peak_prominence=1.5,              # MAD 归一化特征上的峰突出度
    minimum_separation=0.7,           # 同通道候选峰最小间隔，s
    prominence_window=6.0,            # 峰突出度的局部评估窗口，s
    neighbor_radius=4,               # 候选点上下各检查的通道数
    neighbor_support=3,              # 至少需要几个其他通道的支持
    neighbor_tolerance=0.35,          # 邻道到达时刻匹配容差，s
    seed_channels=32,                # 局部 Hough 窗口的通道数
    seed_stride=16,                  # 局部 Hough 窗口移动步长，通道
    seed_minimum_support=12,         # 种子至少支持的通道数
    seed_tolerance=0.25,             # 种子直线与候选峰的匹配容差，s
    measurement_sigma=0.25,          # 观测时刻标准差，s；允许包络峰的局部时刻抖动
    curvature_noise=0.0005,          # 曲率变化的过程噪声强度尺度
    gate_sigma=3.0,                  # 马氏距离门控的标准差倍数
    maximum_time_error=0.8,          # 再额外限制最大时刻残差，s
    maximum_missed_channels=20,      # 允许连续缺测的通道数
    minimum_track_channels=32,       # 一条输出轨迹至少包含的实测通道数
    minimum_density=0.5,             # 支持通道数 / 轨迹跨度通道数
    merge_tolerance=0.4,             # 跨窗口同通道时刻匹配容差，s
    merge_minimum_channels=8,        # 跨窗口至少共同支持的通道数
    merge_fraction=0.7,              # 共同通道中时刻一致的比例
    save_window_figures=True,        # 保存每个窗口的双向轨迹叠加图
    resume=True,                    # 复用同一指纹目录中已完成的窗口
)


# 1. 文件索引与跨文件读取

文件头为 20 个小端 float32；`header[7:10]` 是每通道采样点数、通道数、采样率。按实际点数拼接，文件名时间只用于检查明显断档及显示时钟。时间坐标始终相对整个目录第一份文件的起点。


In [ ]:
def build_index(folder, settings):
    files = sorted(folder.glob('*.bin'))  # 按文件名时间排序的路径
    if not files:
        raise FileNotFoundError(f'目录中没有 BIN 文件：{folder}')
    frames = []  # 每个文件的每通道采样点数
    signatures = []  # 用于判断缓存是否仍有效的文件元数据
    previous_time = None  # 前一个文件名中的起始时间
    for path in files:  # 当前文件路径
        header = np.fromfile(path, dtype='<f4', count=20)  # 80 字节文件头
        if len(header) != 20 or not np.isfinite(header).all():
            raise ValueError(f'无效文件头：{path.name}')
        count, channels, rate = int(header[7]), int(header[8]), float(header[9])
        if count <= 0 or channels <= 0 or rate <= 0 or header[7] != count or header[8] != channels:
            raise ValueError(f'无效的数据规格：{path.name}')
        if frames and (channels != num_channels or rate != sampling_rate):
            raise ValueError(f'文件间通道数或采样率不一致：{path.name}')
        num_channels, sampling_rate = channels, rate  # 全目录统一的数据规格
        stat = path.stat()  # 文件长度及修改时间
        if stat.st_size != 80 + count * channels * 4:
            raise ValueError(f'文件头与数据体长度不一致：{path.name}')
        clock = datetime.strptime(path.name.split('_')[1], '%Y-%m-%d-%H-%M-%S')
        if previous_time is not None:
            gap = (clock - previous_time).total_seconds()  # 文件名起点间隔，s
            if abs(gap - frames[-1] / sampling_rate) > 1.1:
                raise ValueError(f'文件存在明显缺失、重复或时间断档：{path.name}')
        previous_time = clock
        frames.append(count)
        signatures.append([path.name, stat.st_size, stat.st_mtime_ns])
    if not 1 <= settings['channel_from'] <= settings['channel_to'] <= num_channels:
        raise ValueError('通道范围超出文件范围')
    if not 0 < settings['frequency_low'] < settings['frequency_high'] < sampling_rate / 2:
        raise ValueError('带通频率必须位于 0 与 Nyquist 频率之间')
    if not 0 <= settings['minimum_speed_kmh'] < settings['maximum_speed_kmh']:
        raise ValueError('速度范围必须满足 0 <= 下限 < 上限')
    if not 0 < settings['step_seconds'] <= settings['window_seconds']:
        raise ValueError('窗口步长必须大于 0 且不大于窗口长度')
    return dict(files=files, frames=frames, boundaries=np.r_[0, np.cumsum(frames)],
                sampling_rate=sampling_rate, num_channels=num_channels,
                signatures=signatures,
                clock_start=datetime.strptime(files[0].name.split('_')[1], '%Y-%m-%d-%H-%M-%S'))


def read_samples(index, first, last):
    blocks = []  # 与目标区间有交集的文件片段
    for i, path in enumerate(index['files']):
        left = max(first, int(index['boundaries'][i]))  # 当前交集的起点
        right = min(last, int(index['boundaries'][i + 1]))  # 当前交集的终点
        if left < right:
            mapped = np.memmap(path, dtype='<f4', mode='r', offset=80,
                               shape=(index['num_channels'], index['frames'][i]))
            blocks.append(mapped[:, left - index['boundaries'][i]:right - index['boundaries'][i]])
    if not blocks:
        raise ValueError('读取区间为空')
    return np.concatenate(blocks, axis=1)


file_index = build_index(data_dir, options)  # 数据规格及各文件边界
sampling_rate = file_index['sampling_rate']  # 原始采样率，后续不降采样
total_samples = int(file_index['boundaries'][-1])  # 全目录总采样点数
window_samples = round(options['window_seconds'] * sampling_rate)  # 窗口长度，采样点
step_samples = round(options['step_seconds'] * sampling_rate)  # 步长，采样点
windows = []  # 各检测窗口的全局采样区间，左闭右开
first = 0  # 当前窗口的起始采样点
while first < total_samples:
    last = min(first + window_samples, total_samples)
    windows.append((first, last))
    if last == total_samples:
        break
    first += step_samples
print(f"{len(file_index['files'])} 个文件；{total_samples / sampling_rate / 60:.2f} 分钟；{len(windows)} 个窗口")


# 2. 预处理与双方向 f-k 特征

每批窗口共享带通、全通道公共模态去除、目标通道 MAD 归一化。预处理保持原采样率；MAD 使用整批已加载上下文，窗口间可能有轻微尺度差异。目录最前、最后的数据没有完整的真实上下文，会在窗口报告中注明。

通道两侧用边缘值衰减到零，避免反射制造相反斜率。时间渐变仅作用于额外的 5 秒上下文；角度平滑借鉴当前 demo1，过渡区可能同时保留少量两个方向成分。f-k 结果不能视为纯净的单向车辆分离，后面的轨迹斜率仍需约束。


In [ ]:
def normalize_rows(values):
    result = np.empty_like(values, dtype=np.float32)  # 逐通道归一化结果
    for row in range(len(values)):
        trace = values[row].astype(float)  # 当前通道的双精度波形
        centered = trace - np.median(trace)  # 去掉通道中位数
        mad = np.median(np.abs(centered))  # 中位绝对偏差
        if mad > np.finfo(float).eps * max(np.max(np.abs(trace)), 1e-30):
            result[row] = centered / (1.4826 * mad)
        else:
            result[row] = 0
    return result


def preprocess(raw, rate, settings):
    sos = signal.iirfilter(4, [settings['frequency_low'], settings['frequency_high']],
                           fs=rate, btype='bandpass', ftype='butter', output='sos')
    filtered = np.empty_like(raw)  # 全通道带通结果
    for row in range(0, len(raw), 8):
        block = signal.sosfilt(sos, raw[row:row + 8], axis=1)  # 正向带通
        filtered[row:row + 8] = signal.sosfilt(sos, block[:, ::-1], axis=1)[:, ::-1]
    common = np.median(filtered, axis=0)  # 全通道公共波形
    energy = float(np.sum(common.astype(float) ** 2))  # 公共模态能量
    if energy > np.finfo(float).eps:
        for row in range(len(filtered)):
            projection = np.sum(filtered[row].astype(float) * common) / energy
            filtered[row] -= projection * common
    selected = filtered[settings['channel_from'] - 1:settings['channel_to']]  # 目标通道
    return normalize_rows(selected)


def directional_features(values, rate, settings):
    pad = settings['padding_channels']  # 通道扩展量
    time_pad = round(settings['fk_padding'] * rate)  # 时间上下文长度
    padded = np.pad(values, ((pad, pad), (0, 0)), mode='edge')  # 边缘延拓，不反射轨迹
    if pad:
        ramp = np.linspace(0, 1, pad, dtype=np.float32)  # 通道扩展区的线性渐变
        padded[:pad] *= ramp[:, None]
        padded[-pad:] *= ramp[::-1, None]
    if time_pad:
        ramp = (0.5 - 0.5 * np.cos(np.pi * np.arange(time_pad) / time_pad)).astype(np.float32)
        padded[:, :time_pad] *= ramp[None, :]
        padded[:, -time_pad:] *= ramp[::-1][None, :]
    spectrum = fft.fft2(padded)  # 原始二维频谱，两个方向共用
    f = fft.fftfreq(padded.shape[1], 1 / rate).astype(np.float32)  # Hz
    k = fft.fftfreq(padded.shape[0], settings['channel_spacing']).astype(np.float32)  # 周/m
    df = rate / padded.shape[1]  # 频率格点间距
    dk = 1 / (padded.shape[0] * settings['channel_spacing'])  # 波数格点间距
    f_bin, k_bin = f[None, :] / df, k[:, None] / dk  # 无量纲格点坐标
    angle = 2 * k_bin * f_bin / (k_bin ** 2 + f_bin ** 2 + 1e-6)  # 双倍极角的正弦
    maximum_speed = settings['maximum_speed_kmh'] / 3.6  # m/s
    minimum_speed = settings['minimum_speed_kmh'] / 3.6  # m/s
    margin = maximum_speed * np.abs(k[:, None]) - np.abs(f[None, :])  # 速度上界距离
    speed_weight = np.clip(margin / df, 0, 1)  # 速度上界内侧一个频率格点的过渡
    speed_weight = 0.5 - 0.5 * np.cos(np.pi * speed_weight)
    if minimum_speed > 0:
        margin = np.abs(f[None, :]) - minimum_speed * np.abs(k[:, None])
        speed_weight *= 0.5 - 0.5 * np.cos(np.pi * np.clip(margin / df, 0, 1))
    features = {}  # 各方向的归一化检测特征
    for direction, sign in [('increasing', -1), ('decreasing', 1)]:
        weight = np.clip((sign * angle + settings['angular_transition']) /
                         (2 * settings['angular_transition']), 0, 1)
        weight = (0.5 - 0.5 * np.cos(np.pi * weight)) * speed_weight
        weight[0, 0] = 0
        if len(k) % 2 == 0:
            weight[len(k) // 2] = 0
        if len(f) % 2 == 0:
            weight[:, len(f) // 2] = 0
        restored = fft.ifft2(spectrum * weight).real[pad:pad + len(values)].copy()
        if settings['feature_mode'] == 'envelope':
            feature = np.empty_like(restored)  # 平滑包络结果
            sigma = settings['smoothing_seconds'] * rate  # 高斯 sigma，采样点
            kernel = signal.windows.gaussian(2 * int(4 * sigma) + 1, max(sigma, 1e-6))
            kernel /= kernel.sum()
            for row in range(0, len(restored), 8):
                envelope = np.abs(signal.hilbert(restored[row:row + 8], axis=1))
                feature[row:row + 8] = signal.fftconvolve(envelope, kernel[None, :], mode='same', axes=1)
        else:
            feature = restored
        features[direction] = normalize_rows(feature)
    return features


# 3. 候选峰与局部 Hough 种子

候选峰先通过邻道方向一致性验证。Hough 在局部通道块中对斜率和截距投票，再回到真实峰时刻拟合短直线。种子允许使用后续通道的信息，因此这是离线算法。

0 km/h 下限不意味着能检测静止车辆：空间推进模型要求单调通过多个通道；非常慢的车辆若在窗口内没有经过足够通道，也不会满足最小轨迹长度。


In [ ]:
def nearest_peak(peaks, expected):
    if len(peaks) == 0:
        return np.full(np.shape(expected), np.nan)
    indices = np.searchsorted(peaks, expected)  # 预测时刻在有序峰中的插入位置
    left = peaks[np.maximum(indices - 1, 0)]
    right = peaks[np.minimum(indices, len(peaks) - 1)]
    return np.where(np.abs(left - expected) <= np.abs(right - expected), left, right)


def slope_grid(duration, settings, direction, radius):
    smallest = settings['channel_spacing'] / (settings['maximum_speed_kmh'] / 3.6)  # 秒/通道
    largest = duration / max(settings['seed_minimum_support'] - 1, 1)
    if settings['minimum_speed_kmh'] > 0:
        largest = min(largest, settings['channel_spacing'] / (settings['minimum_speed_kmh'] / 3.6))
    step = settings['seed_tolerance'] / max(radius, 1)  # 足以覆盖局部匹配容差的斜率步长
    slopes = np.arange(smallest, max(smallest, largest) + step / 2, step)
    return slopes if direction == 'increasing' else -slopes


def detect_peaks(feature, rate, origin, first, last, direction, settings):
    locations, strengths = [], []  # 每通道的峰时刻和突出度
    begin, end = first - origin, last - origin  # 目标窗口在特征中的列范围
    for trace in feature:
        peaks, properties = signal.find_peaks(
            trace, prominence=settings['peak_prominence'],
            distance=max(1, round(settings['minimum_separation'] * rate)),
            wlen=round(settings['prominence_window'] * rate) | 1)
        valid = (peaks >= begin) & (peaks < end) & (trace[peaks] > 0)
        locations.append((peaks[valid] + origin) / rate)
        strengths.append(properties['prominences'][valid])
    raw_count = sum(map(len, locations))  # 局部验证前候选峰总数
    slopes = slope_grid((last - first) / rate, settings, direction, settings['neighbor_radius'])
    validated, validated_strengths = [], []
    for row, peaks in enumerate(locations):
        support = np.zeros((len(peaks), len(slopes)), dtype=np.int16)
        for neighbor in range(max(0, row - settings['neighbor_radius']),
                              min(len(locations), row + settings['neighbor_radius'] + 1)):
            if neighbor == row:
                continue
            expected = peaks[:, None] + slopes[None, :] * (neighbor - row)
            support += np.abs(nearest_peak(locations[neighbor], expected) - expected) <= settings['neighbor_tolerance']
        keep = np.max(support, axis=1) >= settings['neighbor_support']
        validated.append(peaks[keep])
        validated_strengths.append(strengths[row][keep])
    return validated, validated_strengths, raw_count


def make_seeds(peaks, first_seconds, last_seconds, direction, settings):
    seeds = []  # 经过真实峰验证的局部直线种子
    width = settings['seed_channels']
    slopes = slope_grid(last_seconds - first_seconds, settings, direction, width)
    bin_width = settings['seed_tolerance'] / 2  # Hough 截距格宽，s
    for start in range(0, len(peaks), settings['seed_stride']):
        stop = min(start + width, len(peaks))
        times, rows = [], []  # 当前通道块内的候选点
        for row in range(start, stop):
            times.extend(peaks[row])
            rows.extend([row] * len(peaks[row]))
        if len(times) < settings['seed_minimum_support']:
            continue
        times, rows = np.asarray(times), np.asarray(rows)
        proposals = []  # 当前块的高票数斜率/截距组合
        for slope in slopes:
            intercepts = times - slope * (rows - start)
            bins = np.floor(intercepts / bin_width).astype(np.int64)
            base = int(bins.min())  # 当前截距直方图的起始格点
            histogram = np.bincount(bins - base)  # 截距投票直方图
            counts = np.convolve(histogram, np.ones(5, dtype=int), mode='full')  # 合并相邻格点，允许实际峰时刻抖动
            maxima, _ = signal.find_peaks(np.r_[0, counts, 0], height=settings['seed_minimum_support'], distance=3)
            for idx in maxima - 1:
                proposals.append((int(counts[idx]), float(slope), (base + idx - 2 + 0.5) * bin_width))
        proposals.sort(reverse=True)
        for votes, slope, intercept in proposals:
            matched_rows, matched_times = [], []
            for row in range(start, stop):
                expected = intercept + slope * (row - start)
                observed = float(nearest_peak(peaks[row], np.array(expected)))
                if np.isfinite(observed) and abs(observed - expected) <= settings['seed_tolerance']:
                    matched_rows.append(row)
                    matched_times.append(observed)
            if len(matched_rows) < settings['seed_minimum_support']:
                continue
            fit = np.polyfit(matched_rows, matched_times, 1)  # 真实峰拟合出的斜率和截距
            if fit[0] * slope <= 0 or abs(fit[0]) < settings['channel_spacing'] / (settings['maximum_speed_kmh'] / 3.6):
                continue
            duplicate = False
            for seed in seeds:
                common = np.intersect1d(matched_rows, seed['rows'])
                if len(common) >= settings['merge_minimum_channels']:
                    agreement = np.abs(np.polyval(fit, common) - np.polyval(seed['fit'], common))
                    if np.mean(agreement <= settings['seed_tolerance']) >= settings['merge_fraction']:
                        duplicate = True
                        break
            if not duplicate:
                seeds.append(dict(rows=np.asarray(matched_rows), fit=fit,
                                  start=matched_rows[0], support=len(matched_rows)))
    return sorted(seeds, key=lambda seed: (seed['start'], -seed['support']))


# 4. 多轨迹曲率 KF 与全局关联

每前进一个通道，预测所有活跃轨迹的到达时刻。候选峰通过马氏距离、最大时间误差、运动方向和速度门控后，统一求解线性分配；每条轨迹有独立的“不匹配”选项。缺测时只预测，不生成观测点。

代价使用位置创新及其方差，速度没有被冒充成独立测量。交叉时速度历史通过预测发挥作用，但单个合并峰不能保证分清两个目标。


In [ ]:
def track_window(peaks, strengths, seeds, direction, settings):
    finished, active = [], []  # 已终止与仍活跃的轨迹
    seed_index = 0  # 下一个待检查的种子
    sign = 1 if direction == 'increasing' else -1
    transition = np.array([[1, 1, 0.5], [0, 1, 1], [0, 0, 1]], dtype=float)
    process = settings['curvature_noise'] ** 2 * np.array(
        [[1/20, 1/8, 1/6], [1/8, 1/3, 1/2], [1/6, 1/2, 1]], dtype=float)
    variance = settings['measurement_sigma'] ** 2  # 峰时刻测量方差
    minimum_slope = settings['channel_spacing'] / (settings['maximum_speed_kmh'] / 3.6)
    maximum_slope = np.inf
    if settings['minimum_speed_kmh'] > 0:
        maximum_slope = settings['channel_spacing'] / (settings['minimum_speed_kmh'] / 3.6)
    for row in range(len(peaks)):
        for track in active:
            track['state'] = transition @ track['state']
            track['covariance'] = transition @ track['covariance'] @ transition.T + process
        candidates = peaks[row]  # 当前通道真实检测到的峰时刻
        count = len(candidates)
        costs = np.full((len(active), count + len(active)), 1e6)  # 实测峰列 + 各轨迹自己的缺测列
        gains = []  # 每条轨迹的卡尔曼增益
        for i, track in enumerate(active):
            residual = candidates - track['state'][0]
            innovation_variance = track['covariance'][0, 0] + variance
            gain = track['covariance'][:, 0] / innovation_variance
            gains.append(gain)
            distances = residual ** 2 / innovation_variance  # 平方马氏距离
            updated_slopes = track['state'][1] + gain[1] * residual
            allowed = (distances <= settings['gate_sigma'] ** 2)
            allowed &= np.abs(residual) <= settings['maximum_time_error']
            allowed &= (sign * updated_slopes >= minimum_slope) & (sign * updated_slopes <= maximum_slope)
            costs[i, :count][allowed] = distances[allowed] + np.log(innovation_variance / variance)
            costs[i, count + i] = settings['gate_sigma'] ** 2 + 2  # 自己的缺测选项，避免强制关联
        assignments = {}  # 轨迹序号到候选峰序号的分配
        if active:
            track_indices, peak_indices = linear_sum_assignment(costs)
            for i, j in zip(track_indices, peak_indices):
                if j < count and costs[i, j] < costs[i, count + i]:
                    assignments[int(i)] = int(j)
        remaining = []  # 本通道之后仍允许继续预测的轨迹
        for i, track in enumerate(active):
            if i in assignments:
                j = assignments[i]
                observed = float(candidates[j])
                residual = observed - track['state'][0]
                gain = gains[i]
                track['state'] += gain * residual
                update = np.eye(3)
                update[:, 0] -= gain
                track['covariance'] = update @ track['covariance'] @ update.T + variance * np.outer(gain, gain)
                track['missed'] = 0
                track['rows'].append(row)
                track['times'].append(observed)
                track['strengths'].append(float(strengths[row][j]))
                track['slopes'].append(float(track['state'][1]))
                track['innovations'].append(float(residual))
            else:
                track['missed'] += 1
            if track['missed'] > settings['maximum_missed_channels'] or sign * track['state'][1] < minimum_slope:
                finished.append(track)
            else:
                remaining.append(track)
        active = remaining
        # 先让已有轨迹关联，再用未分配的真实峰启动新轨迹，避免新种子抢走已有车辆的点。
        used = set(assignments.values())  # 本通道已被已有轨迹占用的峰编号
        while seed_index < len(seeds) and seeds[seed_index]['start'] <= row:
            seed = seeds[seed_index]
            seed_index += 1
            slope = float(seed['fit'][0])
            if not minimum_slope <= sign * slope <= maximum_slope or not count:
                continue
            expected = float(np.polyval(seed['fit'], row))
            nearest = int(np.argmin(np.abs(candidates - expected)))
            if nearest in used or abs(candidates[nearest] - expected) > settings['seed_tolerance']:
                continue
            observed = float(candidates[nearest])
            state = np.array([observed, slope, 0.0])
            covariance = np.diag([variance, 0.03 ** 2, 0.005 ** 2])
            active.append(dict(state=state, covariance=covariance, missed=0,
                               rows=[row], times=[observed], strengths=[float(strengths[row][nearest])],
                               slopes=[slope], innovations=[0.0]))
            used.add(nearest)
    finished.extend(active)
    accepted = []  # 满足长度和覆盖率要求的最终局部轨迹
    for track in finished:
        if len(track['rows']) < settings['minimum_track_channels']:
            continue
        density = len(track['rows']) / (track['rows'][-1] - track['rows'][0] + 1)
        if density < settings['minimum_density']:
            continue
        accepted.append(dict(direction=direction, rows=track['rows'], times=track['times'],
                             strengths=track['strengths'], slopes=track['slopes'], density=density,
                             innovation_rmse=float(np.sqrt(np.mean(np.square(track['innovations']))))))
    return accepted


# 5. 跨窗口 ID 匹配、导出和绘图

相邻重叠窗口中的轨迹，只有方向相同、足够多共同通道的实测时刻一致，才允许连接。每个新窗口用一次全局分配，避免把同一窗口的两条轨迹合并为一个 ID。不同方向永不合并。缺少共同观测的片段保持不同 ID，不凭空连接。

每个窗口先保存图和 JSON，再标记完成。参数或 BIN 文件变化会生成新的结果目录；如果修改算法函数，需要同步修改运行函数中的 `algorithm_version`，避免续跑复用旧算法的检查点。JSON、CSV 中的时刻均相对第一份 BIN 文件起点；时钟列按第一份文件名加采样时间换算。预览底图约以 20 Hz 取样以加快保存，检测、匹配及导出仍保持原始采样率。

逐窗口底图和全段观测密度总览都调用 DASPy 的 `plot`，使用 `transpose=True` 保持时间向右、通道号向上的坐标。轨迹实测点、候选峰与编号叠加在 DASPy 返回的坐标轴上。旧窗口检查点缺少 DASPy 绘图标记时，只更新图片，复用原检测结果。


In [ ]:
def write_json(path, value):
    temporary = path.with_suffix(path.suffix + '.tmp')  # 写完后替换，避免留下半个检查点
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def plot_window(features, peaks_by_direction, tracks, first, last, rate, settings, path):
    fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True, sharey=True, layout='constrained')
    for ax, direction in zip(axes, ['increasing', 'decreasing']):
        stride = max(1, round(rate / 20))  # 预览图约每秒显示 20 列；检测和导出仍使用原采样率
        image = features[direction][:, ::stride]  # 只对绘图取样，避免渲染数千万像素
        limit = max(float(np.percentile(np.abs(image), 99)), 1e-8)
        ax = daspy_plot(
            image, fs=rate / stride, ax=ax, xmode='channel', transpose=True,
            t0=first / rate, x0=settings['channel_from'],
            cmap='RdBu_r', vmin=-limit, vmax=limit,
            colorbar_label='Normalized feature')  # DASPy 绘制时间—通道底图，采样率与显示步长匹配
        for row, points in enumerate(peaks_by_direction[direction]):
            ax.scatter(points, np.full(len(points), row + settings['channel_from']), s=2, c='gray', alpha=0.3)
        for i, track in enumerate(tracks):
            if track['direction'] != direction:
                continue
            channels = np.asarray(track['rows']) + settings['channel_from']
            # 散点是实测峰；不把缺测区的模型预测画成观测。
            ax.scatter(track['times'], channels, s=7, label=f"local {i + 1}")
            middle = len(channels) // 2
            ax.text(track['times'][middle], channels[middle], str(i + 1), fontsize=8,
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
        count = sum(t['direction'] == direction for t in tracks)
        ax.set(title=f'{direction}: {count} local tracks', ylabel='Channel', xlim=(first / rate, last / rate))
    axes[-1].set_xlabel('Time from first BIN start [s]')
    fig.savefig(path, dpi=120)
    plt.close(fig)


def plot_overview(tracks, index, settings, path):
    duration = int(index['boundaries'][-1]) / index['sampling_rate']  # 总时长，s
    channels = settings['channel_to'] - settings['channel_from'] + 1  # 目标通道数
    density = np.zeros((channels, max(1, int(np.ceil(duration)))), dtype=np.float32)  # 每通道、每秒的实测峰数量
    for track in tracks:
        for row, point in track['points'].items():
            column = min(int(point['time']), density.shape[1] - 1)  # 观测所属的 1 秒格点
            density[int(row), column] += 1
    fig, ax = plt.subplots(figsize=(18, 6), layout='constrained')
    ax = daspy_plot(
        density, fs=60.0, ax=ax, xmode='channel', transpose=True,
        t0=0, x0=settings['channel_from'], cmap='Greys',
        vmin=0, vmax=max(2, float(density.max()) * 2),
        xlabel='Time from first BIN start [min]', ylabel='Channel',
        colorbar_label='Observed peaks / 1 s bin')  # 时间轴以分钟显示：每分钟 60 个一秒格点
    for track in tracks:
        rows = sorted(track['points'])
        ax.scatter([track['points'][row]['time'] / 60 for row in rows],
                   np.asarray(rows) + settings['channel_from'], s=2,
                   c='#0072B2' if track['direction'] == 'increasing' else '#D55E00')
    ax.set(xlim=(0, duration / 60),
           title=f'Observed tracks: {len(tracks)} IDs; blue=increasing, orange=decreasing')
    fig.savefig(path, dpi=150)
    plt.close(fig)


def merge_windows(records, settings):
    global_tracks = []  # 所有窗口共享的轨迹 ID 及逐通道观测
    for record in records:
        local = record['tracks']
        start = record['first_sample'] / record['sampling_rate']
        eligible = [i for i, track in enumerate(global_tracks)
                    if max(p['time'] for p in track['points'].values()) >= start - settings['merge_tolerance']]
        costs = np.full((len(local), len(eligible) + len(local)), 1e6)
        for i, track in enumerate(local):
            points = dict(zip(track['rows'], track['times']))
            for j, global_index in enumerate(eligible):
                previous = global_tracks[global_index]
                if previous['direction'] != track['direction']:
                    continue
                common = sorted(set(points).intersection(previous['points']))
                if len(common) < settings['merge_minimum_channels']:
                    continue
                residuals = np.array([abs(points[row] - previous['points'][row]['time']) for row in common])
                if np.mean(residuals <= settings['merge_tolerance']) >= settings['merge_fraction']:
                    costs[i, j] = np.median(residuals) / settings['merge_tolerance']
            costs[i, len(eligible) + i] = 2
        assignment = {}
        if local:
            rows, columns = linear_sum_assignment(costs)
            for i, j in zip(rows, columns):
                if j < len(eligible) and costs[i, j] < 2:
                    assignment[int(i)] = eligible[int(j)]
        for i, track in enumerate(local):
            if i in assignment:
                target = global_tracks[assignment[i]]
            else:
                target = dict(track_id=len(global_tracks) + 1, direction=track['direction'], points={}, windows=[])
                global_tracks.append(target)
            target['windows'].append(record['window_index'])
            track['global_id'] = target['track_id']
            for row, observed, strength, slope in zip(track['rows'], track['times'], track['strengths'], track['slopes']):
                # 重叠窗口同通道重复观测，保留突出度较高的一次，不插值和平均峰位置。
                if row not in target['points'] or strength > target['points'][row]['strength']:
                    target['points'][row] = dict(time=observed, strength=strength, slope=slope)
    return global_tracks


def export_results(records, tracks, index, settings, folder):
    summary, observations, window_summary = [], [], []
    for track in tracks:
        rows = sorted(track['points'])
        times = np.array([track['points'][row]['time'] for row in rows])
        speeds = np.array([settings['channel_spacing'] / abs(track['points'][row]['slope']) * 3.6 for row in rows])
        gaps = np.diff(rows) - 1
        summary.append(dict(track_id=track['track_id'], direction=track['direction'],
                            start_seconds=float(times.min()), end_seconds=float(times.max()),
                            start_clock=(index['clock_start'] + timedelta(seconds=float(times.min()))).isoformat(),
                            end_clock=(index['clock_start'] + timedelta(seconds=float(times.max()))).isoformat(),
                            first_channel=rows[0] + settings['channel_from'],
                            last_channel=rows[-1] + settings['channel_from'],
                            observed_channels=len(rows), density=len(rows) / (rows[-1] - rows[0] + 1),
                            maximum_gap_channels=int(gaps.max()) if len(gaps) else 0,
                            median_speed_kmh=float(np.median(speeds)), windows=len(track['windows'])))
        for row, speed in zip(rows, speeds):
            point = track['points'][row]
            observations.append(dict(track_id=track['track_id'], direction=track['direction'],
                                     channel=row + settings['channel_from'], seconds=point['time'],
                                     sample=round(point['time'] * index['sampling_rate']),
                                     clock=(index['clock_start'] + timedelta(seconds=point['time'])).isoformat(),
                                     speed_kmh=float(speed), prominence=point['strength']))
    for record in records:
        window_summary.append(dict(window_index=record['window_index'],
                                   start_seconds=record['first_sample'] / record['sampling_rate'],
                                   end_seconds=record['last_sample'] / record['sampling_rate'],
                                   candidate_peaks=record['candidate_peaks'],
                                   local_tracks=len(record['tracks']),
                                   boundary_context_limited=record['boundary_context_limited'],
                                   seconds=record['runtime_seconds']))
    for filename, values, fields in [
        ('tracks.csv', summary, ['track_id','direction','start_seconds','end_seconds','start_clock','end_clock','first_channel','last_channel','observed_channels','density','maximum_gap_channels','median_speed_kmh','windows']),
        ('observations.csv', observations, ['track_id','direction','channel','seconds','sample','clock','speed_kmh','prominence']),
        ('windows.csv', window_summary, ['window_index','start_seconds','end_seconds','candidate_peaks','local_tracks','boundary_context_limited','seconds'])]:
        with (folder / filename).open('w', newline='', encoding='utf-8-sig') as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            writer.writeheader()
            writer.writerows(values)
    write_json(folder / 'global_tracks.json', tracks)
    write_json(folder / 'report.json', dict(options=settings, files=len(index['files']),
               duration_seconds=int(index['boundaries'][-1]) / index['sampling_rate'],
               windows=len(records), global_tracks=len(tracks),
               notes=['IDs are inferred; vehicle accuracy requires annotated validation.',
                      'Only observed feature peaks are exported; missing values are not interpolated.',
                      'All samples are retained at the original sampling rate.']))
    plot_overview(tracks, index, settings, folder / 'overview.png')
    return summary, window_summary


# 6. 执行全部窗口（可续跑）

这一格处理整个目录，可能需要数分钟。每批共享预处理；每个窗口输出 JSON 和叠加图。不要同时在两个进程中运行同一输出目录。修改参数或源文件后会自动选择新的指纹目录，不复用旧检测结果。


In [ ]:
def run_all_windows(index, windows, settings, root):
    manifest = dict(algorithm_version='demo2-curvature-kf-3', options=settings, files=index['signatures'])
    fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:12]
    folder = root / ('run_' + fingerprint)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'windows').mkdir(exist_ok=True)
    (folder / 'figures').mkdir(exist_ok=True)
    write_json(folder / 'manifest.json', manifest)
    rate = index['sampling_rate']
    total = int(index['boundaries'][-1])
    padding = round(settings['preprocessing_padding'] * rate)
    fk_padding = round(settings['fk_padding'] * rate)
    records = []  # 全部已完成窗口的记录
    started = time.perf_counter()
    print('输出目录：', folder, flush=True)
    for batch_start in range(0, len(windows), settings['windows_per_batch']):
        batch = windows[batch_start:batch_start + settings['windows_per_batch']]
        pending = []  # 本批需要检测或更新图片的窗口编号
        cached = {}  # 可复用的窗口检测结果
        for offset in range(len(batch)):
            number = batch_start + offset
            checkpoint = folder / 'windows' / f'{number:04d}.json'
            if settings['resume'] and checkpoint.exists():
                cached[number] = json.loads(checkpoint.read_text(encoding='utf-8'))
                figure = folder / 'figures' / f'{number:04d}.png'  # 此窗口的轨迹叠加图
                plot_ready = (not settings['save_window_figures'] or
                              (figure.exists() and cached[number].get('figure_backend') == 'daspy-1'))
                if plot_ready:
                    continue
            pending.append(number)
        if pending:
            read_first = max(0, batch[0][0] - padding - fk_padding)
            read_last = min(total, batch[-1][1] + padding + fk_padding)
            raw = read_samples(index, read_first, read_last)
            prepared = preprocess(raw, rate, settings)
            del raw
        for offset, (first, last) in enumerate(batch):
            number = batch_start + offset
            checkpoint = folder / 'windows' / f'{number:04d}.json'
            if number not in pending:
                record = cached[number]
                records.append(record)
                print(f'[{number + 1}/{len(windows)}] 已完成，复用检查点', flush=True)
                continue
            tick = time.perf_counter()
            left, right = max(0, first - fk_padding), min(total, last + fk_padding)
            values = prepared[:, left - read_first:right - read_first]
            missing_left = max(0, fk_padding - first)
            missing_right = max(0, last + fk_padding - total)
            values = np.pad(values, ((0, 0), (missing_left, missing_right)), mode='edge')
            origin = first - fk_padding  # f-k 特征第一列的全局采样编号，目录开头可为负
            features = directional_features(values, rate, settings)
            tracks, peaks_by_direction, visible_features = [], {}, {}
            candidate_count = 0
            seed_count = 0
            for direction in ['increasing', 'decreasing']:
                peaks, strengths, raw_count = detect_peaks(features[direction], rate, origin, first, last, direction, settings)
                if number not in cached:
                    seeds = make_seeds(peaks, first / rate, last / rate, direction, settings)
                    found = track_window(peaks, strengths, seeds, direction, settings)
                    for local_id, track in enumerate(found):
                        track['local_id'] = len(tracks) + local_id + 1
                    tracks.extend(found)
                    seed_count += len(seeds)
                peaks_by_direction[direction] = peaks
                visible_features[direction] = features[direction][:, fk_padding:fk_padding + last - first]
                candidate_count += raw_count
            if number in cached:
                record = cached[number]
                tracks = record['tracks']  # 只更新 DASPy 图片，保留检测点、ID 和原耗时
            else:
                record = dict(window_index=number, first_sample=first, last_sample=last, sampling_rate=rate,
                              tracks=tracks, candidate_peaks=candidate_count, seeds=seed_count,
                              boundary_context_limited=(first - padding - fk_padding < 0 or last + padding + fk_padding > total),
                              runtime_seconds=time.perf_counter() - tick)
            if settings['save_window_figures']:
                plot_window(visible_features, peaks_by_direction, tracks, first, last, rate, settings,
                            folder / 'figures' / f'{number:04d}.png')
                record['figure_backend'] = 'daspy-1'
            write_json(checkpoint, record)
            records.append(record)
            if number in cached:
                print(f'[{number + 1}/{len(windows)}] DASPy 图已更新；检测结果复用；'
                      f'累计 {time.perf_counter() - started:.0f}s', flush=True)
            else:
                print(f'[{number + 1}/{len(windows)}] {first / rate:.1f}–{last / rate:.1f}s；'
                      f'{candidate_count} 候选；{seed_count} 种子；{len(tracks)} 轨迹；'
                      f'累计 {time.perf_counter() - started:.0f}s', flush=True)
            del features, visible_features, values
        if pending:
            del prepared
    global_tracks = merge_windows(records, settings)
    track_summary, window_summary = export_results(records, global_tracks, index, settings, folder)
    return folder, records, global_tracks, track_summary, window_summary


result_dir, window_results, global_tracks, track_summary, window_summary = run_all_windows(
    file_index, windows, options, output_root)
print(f'全部完成：{len(window_results)} 个窗口，{len(global_tracks)} 个关联 ID。结果：{result_dir}')


# 7. 查看汇总与某个窗口

`tracks.csv` 是跨窗口关联后的轨迹汇总，`observations.csv` 是逐点实测峰，`windows.csv` 是全部切片统计。局部图中的编号是该窗口的局部编号；最终全局 ID 在 `global_tracks.json` 与 CSV 中。窗口轨迹数之和含重复，不能直接当作车流量。

CSV 使用实际通道号；`global_tracks.json` 的 `points` 字典键是目标通道内的相对行号，实际通道号为键加 `channel_from`。`windows.csv` 的 `seconds` 是本窗口特征计算和追踪耗时，不含共享预处理和绘图。速度列来自滤波状态，不能与直接差分实测峰得到的速度混为一谈。


In [ ]:
from IPython.display import display, Image

display(Image(filename=str(result_dir / 'overview.png')))  # 全时段方向分布与实测轨迹
preview_window = 0  # 想查看的窗口编号，从 0 开始
preview_path = result_dir / 'figures' / f'{preview_window:04d}.png'  # 对应的轨迹叠加图
if preview_path.exists():
    display(Image(filename=str(preview_path)))
print('轨迹汇总：', result_dir / 'tracks.csv')
print('逐点观测：', result_dir / 'observations.csv')
print('切片统计：', result_dir / 'windows.csv')
